<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/14-structured-streaming/02-output-modes-and-sinks.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 14.2 — Output modes and sinks, side by side

Run the SAME aggregation in `complete` vs `update` mode and diff what
lands in the memory sink each time; run a row-level transform in
`append`; close with `foreachBatch` writing to two destinations from
one query -- ordinary batch DataFrame code, per micro-batch.

In [ ]:
import os, sys, shutil, time
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("14.2-output-modes-sinks")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

In [ ]:
src = (spark.readStream.format("rate").option("rowsPerSecond", 8).load()
       .withColumn("bucket", (col("value") % 5).cast("int")))

## `complete` — the WHOLE result table, every micro-batch

In [ ]:
complete_q = (src.groupBy("bucket").count()
              .writeStream.outputMode("complete")
              .format("memory").queryName("complete_demo").start())

time.sleep(4)
print("complete mode -- always the full 5-bucket table:")
spark.sql("SELECT * FROM complete_demo ORDER BY bucket").show()
complete_q.stop()

## `update` — only rows whose value CHANGED this micro-batch

Memory sink accumulates every emitted row across batches, so we'll see
several rows per bucket over time (one per change), not one final row
each -- that's the visible difference from `complete`.

In [ ]:
update_q = (src.groupBy("bucket").count()
            .writeStream.outputMode("update")
            .format("memory").queryName("update_demo").start())

time.sleep(4)
print("update mode -- one row per CHANGE, accumulated in the sink "
      "(so counts appear multiple times, once per update):")
spark.sql("SELECT bucket, count(*) AS n_updates_seen FROM update_demo GROUP BY bucket ORDER BY bucket").show()
update_q.stop()

## `append` — row-level transform, no aggregation, the natural default

In [ ]:
append_q = (src.filter(col("bucket") == 0)
            .writeStream.outputMode("append")
            .format("memory").queryName("append_demo").start())

time.sleep(3)
print("append mode -- every row emitted exactly once, ever:")
spark.sql("SELECT count(*) AS n FROM append_demo").show()
append_q.stop()

## Rejected: aggregation + `append`, no watermark

14.2's validation rule, confirmed live.

In [ ]:
from pyspark.sql.utils import AnalysisException
try:
    bad = (src.groupBy("bucket").count()
           .writeStream.outputMode("append")
           .format("memory").queryName("bad_append").start())
except AnalysisException as e:
    print("rejected at .start() time:")
    print(str(e).splitlines()[0])

## `foreachBatch` — plain DataFrame code, two destinations at once

In [ ]:
shutil.rmtree("output/fb_parquet", ignore_errors=True)
seen_batches = []

def write_batch(batch_df, batch_id):
    seen_batches.append(batch_id)
    batch_df.write.mode("append").parquet(f"output/fb_parquet/batch_{batch_id}")
    n = batch_df.count()   # ordinary action -- batch_df is a REGULAR DataFrame here
    print(f"  batch {batch_id}: {n} rows, wrote to parquet AND logged this line")

fb_q = (src.writeStream.foreachBatch(write_batch)
        .option("checkpointLocation", "checkpoints/fb").start())

time.sleep(4)
fb_q.stop()
print("batches handled:", seen_batches)
print("parquet dirs written:", os.listdir("output/fb_parquet") if os.path.exists("output/fb_parquet") else [])

In [ ]:
shutil.rmtree("output", ignore_errors=True)
shutil.rmtree("checkpoints", ignore_errors=True)
spark.stop()

## Exercises

1. Run the `complete` demo again but with 50 buckets instead of 5
   (`col("value") % 50`). Watch how much bigger the emitted table is
   each micro-batch. Connect this to the lesson's warning about
   `complete` mode cost scaling with RESULT size.
2. In the `update` demo, query `update_demo` for a SPECIFIC bucket
   and print its count values in the order they were emitted. Are they
   monotonically increasing? Should they be?
3. Try `append` mode on the raw `src` DataFrame (no filter, no
   aggregation) into a `parquet` file sink instead of memory. Confirm
   files land in the output directory as the query runs.
4. Modify `write_batch` to write to a JDBC sink or a second parquet
   path as well as the first, demonstrating "multiple destinations
   from one query" concretely.
5. Remove the `checkpointLocation` option from the `foreachBatch`
   query and see what happens at `.start()`. Does it behave differently
   than omitting it from a `console` sink query? Why?

In [ ]:
# your exercise code here